In [1]:
print("faris")

faris


In [2]:
# za svaki bat i:                     # prolazimo kroz svakog šišmiša pojedinačno
#                                     # i = indeks jednog šišmiša u populaciji
 
#     beta = random(0, 1)             # slučajan broj između 0 i 1
#                                     # koristi se da frekvencija bude malo drugačija za svakog šišmiša
 
#     f[i] = f_min + (f_max - f_min) * beta
#                                     # f[i] = frekvencija i-tog šišmiša
#                                     # frekvencija određuje koliko jako mijenja svoje kretanje
#                                     # nije "pozicija", nego parametar koji utiče na pomak
 
#     v[i] = v[i] + (x[i] - best) * f[i]
#                                     # v[i] = brzina i-tog šišmiša
#                                     # brzina govori koliko i u kojem smjeru će se pomjeriti
#                                     # x[i] = trenutna pozicija tog šišmiša
#                                     # best = trenutno najbolje rješenje od svih šišmiša
#                                     # ovom formulom šišmiš koriguje svoju brzinu u odnosu na best
 
#     x_new = x[i] + v[i]
#                                     # x_new = nova kandidatska pozicija za tog jednog šišmiša
#                                     # znači: uzmemo staru poziciju i dodamo brzinu
#                                     # DA — ovo je nova pozicija JEDNOG šišmiša, ovog i-tog
 
#     ako random(0,1) > pulse_rate[i]:
#         x_new = best + epsilon * average_loudness
#                                     # ponekad šišmiš ne ide običnim pomakom
#                                     # nego skoči blizu trenutno najboljeg rješenja
#                                     # epsilon = mali slučajni broj / slučajan mali pomak
#                                     # average_loudness = prosječna glasnoća svih šišmiša
#                                     # ovo služi za lokalnu pretragu oko najboljeg rješenja
 
#     x_new = popravi_granice(x_new)
#                                     # ako je nova pozicija izašla van dozvoljenog opsega,
#                                     # vrati je unutar granica problema
 
#     fitness_new = objective(x_new)
#                                     # izračunaj koliko je dobra nova pozicija
#                                     # objective = funkcija koju minimiziraš ili maksimiziraš
 
#     ako fitness_new < fitness[i] I random(0,1) < loudness[i]:
#         x[i] = x_new
#                                     # prihvati novu poziciju za tog šišmiša
 
#         fitness[i] = fitness_new
#                                     # zapamti novu vrijednost funkcije za tog šišmiša
 
#         loudness[i] = alpha * loudness[i]
#                                     # smanji glasnoću tog šišmiša
#                                     # što iteracije više idu, šišmiš postaje "mirniji"

In [3]:
OPCIJE = {
    # tt_split, random_state, C, kernel, gamma
    # "tt_split": [0.1, 0.2, 0.22, 0.25, 0.29, 0.3, 0.33, 0.35, 0.4, 0.45, 0.5],
    "tt_split": [0.2, 0.22, 0.25, 0.29],
    # "tt_split": np.arange(0.1, 0.5, 0.0001).tolist(),
    "random_state": [0, 1, 2, 3],
    "C": [0.1, 0.5, 1, 2, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"]
}


# "tt_split": np.arange(0.32, 0.34, 0.0001).tolist(),

# SVM - Support Vector Machine - algoritam za klasifikaciju i regresiju

# RandomSearchCV - metoda za pronalaženje najboljih hiperparametara modela
# GridSearchCV - metoda za pronalaženje najboljih hiperparametara modela - isprobava sve kombinacije


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import f1_score
import random
import numpy as np

In [5]:
# Učitavanje podataka
df = pd.read_csv("iris.csv") 

X = df.drop('species', axis=1)
y = df['species']

In [6]:
X.head()

,sepal_length,sepal_width,petal_length,petal_width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


In [7]:
y.head()

0    setosa
1    setosa
2    setosa
3    setosa
4    setosa
Name: species, dtype: object

In [8]:
# Ovaj dio koda radi evaluaciju eksperimenata dati u varijabli OPCIJE.
# Za svaki eksperiment, dijeli podatke na trening i test skup, trenira SVM model sa zadanim hiperparametrima, i računa F1 score na test skupu.
# Koristeci GridSearchCV ili RandomSearchCV bi bilo efikasnije, ali ovaj kod demonstrira osnovni pristup evaluacije.

# Najbolji parametri: 
best_params = {
    "tt_split": 0.29,
    "random_state": 0,
    "C": 0.1,
    "kernel": "linear",
    "gamma": "scale"
}

best_score = 0
best_model = None

for tt_split in OPCIJE["tt_split"]:
    for random_state in OPCIJE["random_state"]:
        for C in OPCIJE["C"]:
            for kernel in OPCIJE["kernel"]:
                for gamma in OPCIJE["gamma"]:

                    # Podjela podataka na trening i test skup
                    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=tt_split, random_state=random_state)

                    # Treniranje SVM modela
                    model = SVC(C=C, kernel=kernel, gamma=gamma)
                    model.fit(X_train, y_train)

                    # Predviđanje na test skupu
                    y_pred = model.predict(X_test)

                    # Računanje F1 score
                    score = f1_score(y_test, y_pred, average='weighted')
                    # ovdje moze ici bilo koja funkcija EVALUACIJE, npr. accuracy_score, precision_score, recall_score, itd.

                    # Ispis rezultata za trenutne parametre
                    print(f"tt_split: {tt_split}, random_state: {random_state}, C: {C}, kernel: {kernel}, gamma: {gamma} => F1 Score: {score}")
                    
                    if score > best_score:
                        best_score = score
                        best_params = {
                            "tt_split": tt_split,
                            "random_state": random_state,
                            "C": C,
                            "kernel": kernel,
                            "gamma": gamma
                        }
                        best_model = model
                    
                    
            

tt_split: 0.2, random_state: 0, C: 0.1, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.1, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.1, kernel: rbf, gamma: scale => F1 Score: 0.8380018674136321
tt_split: 0.2, random_state: 0, C: 0.1, kernel: rbf, gamma: auto => F1 Score: 0.9672820512820512
tt_split: 0.2, random_state: 0, C: 0.5, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: rbf, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 0.5, kernel: rbf, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: linear, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: linear, gamma: auto => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: rbf, gamma: scale => F1 Score: 1.0
tt_split: 0.2, random_state: 0, C: 1, kernel: rbf,

In [9]:
best_params, best_score


({'tt_split': 0.2,
  'random_state': 0,
  'C': 0.1,
  'kernel': 'linear',
  'gamma': 'scale'},
 1.0)

In [10]:
best_model


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",0.1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [11]:
# RandomSearch Implementacija
from sklearn.model_selection import RandomizedSearchCV

# ValueError: Invalid parameter 'tt_split' for estimator SVC(). Valid parameters are: ['C', 'break_ties', 'cache_size', 'class_weight', 'coef0', 'decision_function_shape', 'degree', 'gamma', 'kernel', 'max_iter', 'probability', 'random_state', 'shrinking', 'tol', 'verbose'].

OPCIJE = {
    "random_state": [0, 1, 2, 3],
    "C": [0.1, 0.5, 1, 2, 10],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"],
    "break_ties": [True, False],
    "cache_size": [200, 300, 400],
    "class_weight": [None, 'balanced'],
    "coef0": [0.0, 0.1, 0.5, 1.0],
    "decision_function_shape": ['ovo', 'ovr'],
    "degree": [3, 4, 5],
    "max_iter": [-1, 100, 200],
    "probability": [True, False],
    "shrinking": [True, False],
    "tol": [1e-3, 1e-4, 1e-5],
    "verbose": [0, 1]
}

random_search = RandomizedSearchCV(estimator=SVC(), param_distributions=OPCIJE, n_iter=100, scoring='f1_weighted', random_state=0)
random_search.fit(X, y)
random_search.best_params_, random_search.best_score_



[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_scorer.py", line 409, in _score
    y_pred = method_caller(
     

[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_scorer.py", line 409, in _score
    y_pred = method_caller(
     

[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]

...

[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=100).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)


[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [0.919288   0.97285627        nan 0.96661654        nan        nan
 0.97285627 0.919288   0.98663325        nan        nan 0.97994987
 0.9732665  0.97994987 0.96661654 0.97994987 0.97994987 0.96661654
 0.96661654 0.97994987 0.97994987 0.97994987 0.97994987 0.98663325
        nan 0.96661654 0.97994987 0.96661654        nan        nan
 0.97994987 0.97285627 0.97994987 0.97994987 0.97994987        nan
 0.97994987 0.94659983 0.97994987 0.97994987 0.95996658 0.97994987
 0.97994987 0.97994987 0.97994987 0.97994987 0.97994987        nan
 0.97994987 0.97994987        nan        nan        nan 0.9732665
        nan        nan 0.97994987 0.97994987 0.97994987 0.97994987
        nan 0.97994987 0.97994987 0.95996658 0.97994987 0.919288
        nan 0.97994987 0.97994987 0.97994987        nan        nan
 0.97994987 0.97994987 0.9

({'verbose': 0,
  'tol': 0.001,
  'shrinking': False,
  'random_state': 1,
  'probability': True,
  'max_iter': 100,
  'kernel': 'linear',
  'gamma': 'scale',
  'degree': 5,
  'decision_function_shape': 'ovr',
  'coef0': 0.0,
  'class_weight': None,
  'cache_size': 400,
  'break_ties': True,
  'C': 0.5},
 0.9866332497911445)

In [12]:
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import f1_score
from sklearn.model_selection import cross_val_score

# =============================================
# ALGORITAM SISMISA (Bat Algorithm) za SVM
# =============================================
# Optimizujemo parametre SVM-a:
#   x[0] = C        (regularizacioni parametar, opseg [0.01, 10])
#   x[1] = gamma    (kernel koeficijent rbf, opseg [0.001, 1])

# --- Parametri algoritma sismisa ---
n_bats       = 20      # broj sismisa u populaciji
max_iter     = 50      # broj iteracija
f_min, f_max = 0, 2   # min/max frekvencija
alpha        = 0.9     # koeficijent smanjenja glasnoce
r0           = 0.5     # pocetni pulse rate
A0           = 1.0     # pocetna glasnoca

# Granice pretrazivanja: [C_min, C_max], [gamma_min, gamma_max]
lower = np.array([0.01, 0.001])
upper = np.array([10.0, 1.0])
dim   = 2  # broj dimenzija (C i gamma)

# --- Fitness funkcija ---
def fitness(x):
    C_val     = float(np.clip(x[0], lower[0], upper[0]))
    gamma_val = float(np.clip(x[1], lower[1], upper[1]))
    model = SVC(C=C_val, kernel='rbf', gamma=gamma_val)
    scores = cross_val_score(model, X, y, cv=5, scoring='f1_weighted')
    return scores.mean()

# --- Inicijalizacija populacije ---
np.random.seed(42)
positions  = np.random.uniform(lower, upper, (n_bats, dim))  # pozicije sismisa
velocities = np.zeros((n_bats, dim))                          # brzine sismisa
loudness   = np.full(n_bats, A0)                              # glasnoce
pulse_rate = np.full(n_bats, r0)                              # pulse rate

# Izracunaj fitness za sve sismise
fitness_vals = np.array([fitness(p) for p in positions])

# Pronadji globalno najbolje rjesenje
best_idx  = np.argmax(fitness_vals)
best_pos  = positions[best_idx].copy()
best_fit  = fitness_vals[best_idx]

print(f"Pocetni best fitness: {best_fit:.4f}, C={best_pos[0]:.4f}, gamma={best_pos[1]:.4f}")
print("Pokrecam Bat Algorithm...")

# --- Glavna petlja ---
for t in range(max_iter):

    for i in range(n_bats):

        # za svaki bat i:
        # Korak 1: Azuriraj frekvenciju za ovog sismisa
        beta = np.random.rand()
        f_i  = f_min + (f_max - f_min) * beta          # f[i] = f_min + (f_max - f_min) * beta

        # Korak 2: Azuriraj brzinu
        # v[i] = v[i] + (x[i] - best) * f[i]
        velocities[i] = velocities[i] + (positions[i] - best_pos) * f_i

        # Korak 3: Generiši novu kandidatsku poziciju
        # x_new = x[i] + v[i]
        x_new = positions[i] + velocities[i]

        # Korak 4: Lokalna pretraga — ako random > pulse_rate skoči blizu best
        # ako random(0,1) > pulse_rate[i]:
        #     x_new = best + epsilon * average_loudness
        if np.random.rand() > pulse_rate[i]:
            avg_loudness = np.mean(loudness)
            epsilon = np.random.uniform(-1, 1, dim)
            x_new = best_pos + epsilon * avg_loudness

        # Korak 5: Popravi granice (x_new = popravi_granice(x_new))
        x_new = np.clip(x_new, lower, upper)

        # Korak 6: Izracunaj fitness nove pozicije (fitness_new = objective(x_new))
        fit_new = fitness(x_new)

        # Korak 7: ako fitness_new < fitness[i] I random(0,1) < loudness[i]:
        # (ovdje maximiziramo F1 pa koristimo >)
        if fit_new > fitness_vals[i] and np.random.rand() < loudness[i]:

            # x[i] = x_new
            positions[i]    = x_new
            # fitness[i] = fitness_new
            fitness_vals[i] = fit_new

            # loudness[i] = alpha * loudness[i]  (smanji glasnocu)
            loudness[i]   = alpha * loudness[i]
            # povecaj pulse rate tokom iteracija
            pulse_rate[i] = r0 * (1 - np.exp(-0.9 * t))

        # Korak 8: Azuriraj globalno best
        if fitness_vals[i] > best_fit:
            best_pos = positions[i].copy()
            best_fit = fitness_vals[i]

    if (t + 1) % 10 == 0:
        print(f"Iteracija {t+1}/{max_iter} — Best F1: {best_fit:.4f}, C={best_pos[0]:.4f}, gamma={best_pos[1]:.4f}")

# --- Rezultati ---
print(f"\n=== Bat Algorithm - Rezultati ===")
print(f"Najbolji C:     {best_pos[0]:.4f}")
print(f"Najbolji gamma: {best_pos[1]:.4f}")
print(f"Najbolji F1:    {best_fit:.4f}")

# Treniraj finalni model sa najboljim parametrima
best_bat_model = SVC(C=best_pos[0], kernel='rbf', gamma=best_pos[1])
best_bat_model.fit(X, y)
print(f"Finalni model treniran.")


Pocetni best fitness: 0.9866, C=0.5903, gamma=0.8663
Pokrecam Bat Algorithm...
Iteracija 10/50 — Best F1: 0.9866, C=0.5903, gamma=0.8663
Iteracija 20/50 — Best F1: 0.9866, C=0.5903, gamma=0.8663
Iteracija 30/50 — Best F1: 0.9866, C=0.5903, gamma=0.8663
Iteracija 40/50 — Best F1: 0.9866, C=0.5903, gamma=0.8663
Iteracija 50/50 — Best F1: 0.9866, C=0.5903, gamma=0.8663

=== Bat Algorithm - Rezultati ===
Najbolji C:     0.5903
Najbolji gamma: 0.8663
Najbolji F1:    0.9866
Finalni model treniran.


In [13]:
# Usporedi Random Search i Bat Algorithm rezultate

print("=" * 55)
print("USPOREDBA METODA")
print("=" * 55)
print(f"\nRandom Search (sklearn RandomizedSearchCV):")
print(f"  Best F1:  {random_search.best_score_:.4f}")
print(f"  Params:   {random_search.best_params_}")
print(f"\nBat Algorithm (custom implementacija):")
print(f"  Best F1:  {best_fit:.4f}")
print(f"  C:        {best_pos[0]:.4f}")
print(f"  gamma:    {best_pos[1]:.4f}")


USPOREDBA METODA

Random Search (sklearn RandomizedSearchCV):
  Best F1:  0.9866
  Params:   {'verbose': 0, 'tol': 0.001, 'shrinking': False, 'random_state': 1, 'probability': True, 'max_iter': 100, 'kernel': 'linear', 'gamma': 'scale', 'degree': 5, 'decision_function_shape': 'ovr', 'coef0': 0.0, 'class_weight': None, 'cache_size': 400, 'break_ties': True, 'C': 0.5}

Bat Algorithm (custom implementacija):
  Best F1:  0.9866
  C:        0.5903
  gamma:    0.8663


In [14]:
OPCIJE = {
    "random_state": np.arange(0, 50).tolist(),
    "C": np.arange(0.1, 10.1, 0.1).tolist(),
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"],
    "break_ties": [True, False],
    "cache_size": [200, 300, 400],
    "class_weight": [None, 'balanced'],
    "coef0": [0.0, 0.1, 0.5, 1.0],
    "decision_function_shape": ['ovo', 'ovr'],
    "degree": [3, 4, 5],
    "max_iter": [-1, 100, 200],
    "probability": [True, False],
    "shrinking": [True, False],
    "tol": [1e-3, 1e-4, 1e-5],
    "verbose": [0, 1]
    #sample
}

random_search = RandomizedSearchCV(estimator=SVC(), param_distributions=OPCIJE, n_iter=100, scoring='f1_weighted', random_state=0)
random_search.fit(X, y)
random_search.best_params_, random_search.best_score_

[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_scorer.py", line 409, in _score
    y_pred = method_caller(
     

[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\Use

[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]

...

[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)


[LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM][LibSVM]

c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\svm\_base.py:313: ConvergenceWarning: Solver terminated early (max_iter=200).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
c:\Users\User\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_search.py:1137: UserWarning: One or more of the test scores are non-finite: [0.97984861        nan        nan 0.97994987        nan 0.97994987
 0.97994987 0.97994

({'verbose': 0,
  'tol': 0.001,
  'shrinking': True,
  'random_state': 35,
  'probability': True,
  'max_iter': 100,
  'kernel': 'rbf',
  'gamma': 'scale',
  'degree': 4,
  'decision_function_shape': 'ovo',
  'coef0': 1.0,
  'class_weight': 'balanced',
  'cache_size': 300,
  'break_ties': False,
  'C': 6.5},
 0.9866332497911445)